# CGM pattern mining and hypoglycemia prediction

## Student Data Analysis Notebook

**Course:** I 320D - Data Science for Biomedical Informatics  
**Instructor:** Ammar Darkazanli  
**Semester:** Spring 2026

---

## Project Goal
Mine CGM time series for temporal patterns associated with hypoglycemia onset. Build a 30-minute ahead binary classifier using shapelet-based features and compare against LSTM baseline.

### Notebook Sections (CORRECT ORDER)
| Part | Topic | Key Methods |
|------|-------|----------|
| 1 | Setup, Load & Validate Data | XML → CSV, duplicate detection, time alignment |
| 2 | Vertical Stacking | Align meal, exercise with CGM timeline |
| 3 | Hypoglycemia Labeling | Label events < 70 mg/dL, create 30-min lookahead windows |
| 4 | Feature Aggregation | Patient-level and temporal feature engineering |
| 5 | Shapelet Discovery & Feature Engineering | Extract discriminative subsequences |
| 6 | Model Evaluation | Binary classifier comparison, lead-time analysis, per-patient AUROC |

---

## Key Clinical Questions to Keep in Mind
- **Physiological Lag:** CGM is interstitial (5-15 min lag behind blood glucose)
- **Class Imbalance:** Hypoglycemia is rare — what % of 5-min windows are < 70 mg/dL?
- **Shapelet Interpretation:** What does a "pre-hypoglycemia shapelet" look like?
- **Personalization:** Should we train one global model or per-patient models?
- **Prevented Events:** How do we handle events averted by patient intervention?

---
# PART 1: Setup and Load Data
---

### 1.1 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xml.etree.ElementTree as ET
from pathlib import Path
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("Libraries Imported!")

Libraries Imported!


### 1.2 Load the OhioT1DM Dataset

In [14]:
# Convert all OhioT1DM XML files to CSV
source_dir = Path(r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project\OhioT1DM\2020\train")
output_dir = Path(r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project")

output_dir.mkdir(parents=True, exist_ok=True)

event_types = {
    'cgm': ['ts', 'value'],
    'finger': ['ts', 'value'],
    'basal': ['ts', 'value', 'duration'],
    'bolus': ['ts', 'dose', 'bwz_carb_input'],
    'meal': ['ts', 'carbs'],
    'sleep': ['ts', 'quality'],
    'exercise': ['ts', 'intensity', 'duration']
}

tag_mapping = {
    'glucose_level': 'cgm',
    'finger_stick': 'finger',
    'basal': 'basal',
    'bolus': 'bolus',
    'meal': 'meal',
    'sleep': 'sleep',
    'exercise': 'exercise'
}

all_data = {key: [] for key in event_types.keys()}

xml_files = sorted(source_dir.glob('*.xml'))
print(f"Found {len(xml_files)} patient files")
print(f"Output directory: {output_dir}\n")

for i, xml_file in enumerate(xml_files, 1):
    try:
        tree = ET.parse(str(xml_file))
        root = tree.getroot()
        patient_id = xml_file.stem
        
        for xml_tag, event_type in tag_mapping.items():
            for parent in root.iter(xml_tag):
                for event in parent.iter('event'):
                    row = {'patient_id': patient_id}
                    for attr in event_types[event_type]:
                        row[attr] = event.get(attr)
                    all_data[event_type].append(row)
        
        if i % 2 == 0:
            print(f"  Processed {i}/{len(xml_files)} files...")
    
    except Exception as e:
        print(f"  Error parsing {xml_file.name}: {e}")

print(f"\nProcessed all {len(xml_files)} files\n")

for event_type, records in all_data.items():
    if records:
        df = pd.DataFrame(records)
        df['ts'] = pd.to_datetime(df['ts'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
        for col in df.columns:
            if col not in ['patient_id', 'ts']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df = df.sort_values('ts').reset_index(drop=True)
        output_file = output_dir / f"{event_type}.csv"
        df.to_csv(output_file, index=False)
        print(f"✓ {event_type}.csv: {len(df):,} records from {df['patient_id'].nunique()} patients")
    else:
        print(f"✗ {event_type}.csv: No data")

print(f"\n✅ Conversion complete!")
print(f"CSV files saved to:\n{output_dir}")

Found 6 patient files
Output directory: C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project

  Processed 2/6 files...
  Processed 4/6 files...
  Processed 6/6 files...

Processed all 6 files

✓ cgm.csv: 65,535 records from 6 patients
✓ finger.csv: 1,691 records from 6 patients
✓ basal.csv: 357 records from 6 patients
✓ bolus.csv: 1,568 records from 6 patients
✓ meal.csv: 702 records from 6 patients
✓ sleep.csv: 150 records from 5 patients
✓ exercise.csv: 61 records from 4 patients

✅ Conversion complete!
CSV files saved to:
C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project


In [3]:
def load_ohiot1dm_csv(csv_dir):
    """Load OhioT1DM data from pre-converted CSV files."""
    
    event_types = ['cgm', 'finger', 'basal', 'bolus', 'meal', 'sleep', 'exercise']
    data = {}
    
    csv_path = Path(csv_dir)
    print(f"Loading CSV files from: {csv_path}\n")
    
    for event_type in event_types:
        csv_file = csv_path / f"{event_type}.csv"
        
        if csv_file.exists():
            try:
                df = pd.read_csv(csv_file)
                
                if 'ts' in df.columns:
                    df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
                
                for col in df.columns:
                    if col not in ['patient_id', 'ts']:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                
                df = df.sort_values('ts').reset_index(drop=True)
                
                data[event_type] = df
                print(f"✓ {event_type}.csv: {len(df):,} records from {df['patient_id'].nunique()} patients")
            
            except Exception as e:
                print(f"✗ {event_type}.csv: Error loading - {e}")
                data[event_type] = pd.DataFrame()
        else:
            print(f"✗ {event_type}.csv: File not found")
            data[event_type] = pd.DataFrame()
    
    return data

csv_dir = r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project"
data = load_ohiot1dm_csv(csv_dir)

cgm = data['cgm']
print(f"\n{'='*60}")
print(f"CGM Data Loaded: {cgm.shape[0]:,} glucose readings")
print(f"{'='*60}")
print("\nFirst 5 rows:")
print(cgm.head())

Loading CSV files from: C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project

✓ cgm.csv: 65,535 records from 6 patients
✓ finger.csv: 1,691 records from 6 patients
✓ basal.csv: 357 records from 6 patients
✓ bolus.csv: 1,568 records from 6 patients
✓ meal.csv: 702 records from 6 patients
✓ sleep.csv: 150 records from 5 patients
✓ exercise.csv: 61 records from 4 patients

CGM Data Loaded: 65,535 glucose readings

First 5 rows:
        patient_id                  ts  value
0  552-ws-training 2025-04-16 11:17:05     95
1  552-ws-training 2025-04-16 11:22:05     86
2  552-ws-training 2025-04-16 11:27:05     81
3  552-ws-training 2025-04-16 11:32:05     81
4  552-ws-training 2025-04-16 11:37:05     82


### 1.3 Data Validation

In [4]:
def validate_ohiot1dm(data):
    """Comprehensive data validation for OhioT1DM datasets."""
    
    print("="*70)
    print("OhioT1DM DATA VALIDATION REPORT")
    print("="*70)
    
    print("\nDATASET OVERVIEW")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            print(f"{event_type:12} | Rows: {len(df):>10,} | Patients: {df['patient_id'].nunique():>3} | "
                  f"Date Range: {df['ts'].min().date()} to {df['ts'].max().date()}")
        else:
            print(f"{event_type:12} | No data")
    
    print("\n\nMISSING VALUES")
    print("-" * 70)
    has_missing = False
    for event_type, df in data.items():
        if not df.empty:
            missing = df.isnull().sum()
            if missing.sum() > 0:
                has_missing = True
                missing_pct = (missing / len(df) * 100).round(2)
                print(f"\n{event_type.upper()}:")
                for col in missing[missing > 0].index:
                    print(f"  {col:20} | Missing: {missing[col]:>6} ({missing_pct[col]:>5.1f}%)")
    
    if not has_missing:
        print("No missing values detected!")
    
    print("\n\nVALUE RANGES & OUTLIERS")
    print("-" * 70)
    
    if not data['cgm'].empty:
        print("\nCGM (Blood Glucose):")
        cgm_vals = data['cgm']['value'].dropna()
        print(f"  Range: {cgm_vals.min():.1f} - {cgm_vals.max():.1f} mg/dL")
        print(f"  Mean:  {cgm_vals.mean():.1f} mg/dL")
        print(f"  Median: {cgm_vals.median():.1f} mg/dL")
        low = (cgm_vals < 20).sum()
        high = (cgm_vals > 600).sum()
        if low > 0 or high > 0:
            print(f"  IMPLAUSIBLE: {low} readings < 20 mg/dL, {high} readings > 600 mg/dL")
        else:
            print(f"  ✓ All values in plausible range")
    
    if not data['finger'].empty:
        print("\nFinger Stick (Reference Glucose):")
        finger_vals = data['finger']['value'].dropna()
        print(f"  Range: {finger_vals.min():.1f} - {finger_vals.max():.1f} mg/dL")
        print(f"  Mean:  {finger_vals.mean():.1f} mg/dL")
        print(f"  Count: {len(finger_vals):,} measurements")
    
    if not data['meal'].empty:
        print("\nMeal (Carbohydrates):")
        meal_vals = data['meal']['carbs'].dropna()
        if len(meal_vals) > 0:
            print(f"  Range: {meal_vals.min():.1f} - {meal_vals.max():.1f} grams")
            print(f"  Mean:  {meal_vals.mean():.1f} grams")
            print(f"  Count: {len(meal_vals):,} meals")
    
    print("\n\nDUPLICATE RECORDS")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            dups = df.duplicated().sum()
            ts_patient_dups = df[['patient_id', 'ts']].duplicated().sum()
            
            if dups > 0 or ts_patient_dups > 0:
                print(f"{event_type:12} | Exact duplicates: {dups:>6} | "
                      f"Same timestamp+patient: {ts_patient_dups:>6}")
            else:
                print(f"{event_type:12} | No duplicates")
    
    print("\n" + "="*70)

validate_ohiot1dm(data)

OhioT1DM DATA VALIDATION REPORT

DATASET OVERVIEW
----------------------------------------------------------------------
cgm          | Rows:     65,535 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
finger       | Rows:      1,691 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
basal        | Rows:        357 | Patients:   6 | Date Range: 2025-04-16 to 2027-06-22
bolus        | Rows:      1,568 | Patients:   6 | Date Range: NaT to NaT
meal         | Rows:        702 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
sleep        | Rows:        150 | Patients:   5 | Date Range: NaT to NaT
exercise     | Rows:         61 | Patients:   4 | Date Range: 2025-04-17 to 2027-07-03


MISSING VALUES
----------------------------------------------------------------------

BASAL:
  duration             | Missing:    357 (100.0%)

BOLUS:
  ts                   | Missing:   1568 (100.0%)
  bwz_carb_input       | Missing:   1568 (100.0%)

SLEEP:
  ts                   | Missing:   

### 1.4 Data Cleaning & Prepare Datasets

In [5]:
# Step 1: Handle duplicate CGM timestamps
cgm_clean = data['cgm'].copy()

duplicates = cgm_clean[cgm_clean.duplicated(subset=['patient_id', 'ts'], keep=False)].sort_values(['patient_id', 'ts'])
print(f"Found {len(duplicates)} records with duplicate patient-timestamp pairs")
print(f"Unique duplicate patient-timestamp pairs: {duplicates[['patient_id', 'ts']].drop_duplicates().shape[0]}\n")

cgm_clean = cgm_clean.groupby(['patient_id', 'ts'], as_index=False).agg({
    'value': 'mean'
})

print(f"CGM after deduplication: {len(cgm_clean):,} records")
print(f"Patients: {cgm_clean['patient_id'].nunique()}")
print(f"Date range: {cgm_clean['ts'].min()} to {cgm_clean['ts'].max()}\n")

# Step 2: Investigate bolus/sleep timestamp issue
print("="*70)
print("INVESTIGATING BOLUS & SLEEP TIMESTAMP ISSUE")
print("="*70)

bolus = data['bolus'].copy()
sleep = data['sleep'].copy()

print(f"\nBOLUS: {len(bolus)} records")
print(f"  Missing timestamps (NaT): {bolus['ts'].isna().sum()}")
print(f"  Valid timestamps: {bolus['ts'].notna().sum()}")

print(f"\nSLEEP: {len(sleep)} records")
print(f"  Missing timestamps (NaT): {sleep['ts'].isna().sum()}")
print(f"  Valid timestamps: {sleep['ts'].notna().sum()}\n")

# Step 3: Prepare datasets_to_use
print("="*70)
print("PART 1 COMPLETE: DATA CLEANED AND READY")
print("="*70)

datasets_to_use = {
    'cgm': cgm_clean,
    'finger': data['finger'].copy(),
    'meal': data['meal'].copy(),
    'basal': data['basal'].copy(),
    'exercise': data['exercise'].copy()
}

print(f"\nDatasets ready for analysis:")
for name, df in datasets_to_use.items():
    if len(df) > 0:
        print(f"  ✓ {name:12} | {len(df):>7,} records | {df['ts'].min()} to {df['ts'].max()}")

print(f"\nDatasets excluded (parsing issues):")
print(f"  ✗ bolus       | All timestamps missing (NaT) - skip for now")
print(f"  ✗ sleep       | All timestamps missing (NaT) - skip for now")

print(f"\n✅ Part 1 complete! Ready for Part 2: Vertical Stacking")
print(f"\nKey stats:")
print(f"  - {cgm_clean['patient_id'].nunique()} patients")
print(f"  - {len(cgm_clean):,} total 5-min readings")
print(f"  - {(cgm_clean['value'] < 70).sum():,} readings < 70 mg/dL (hypoglycemic)")
print(f"  - {(cgm_clean['value'] < 70).sum() / len(cgm_clean) * 100:.2f}% hypoglycemic prevalence")

Found 0 records with duplicate patient-timestamp pairs
Unique duplicate patient-timestamp pairs: 0

CGM after deduplication: 65,535 records
Patients: 6
Date range: 2025-04-16 11:17:05 to 2027-07-03 23:56:44

INVESTIGATING BOLUS & SLEEP TIMESTAMP ISSUE

BOLUS: 1568 records
  Missing timestamps (NaT): 1568
  Valid timestamps: 0

SLEEP: 150 records
  Missing timestamps (NaT): 150
  Valid timestamps: 0

PART 1 COMPLETE: DATA CLEANED AND READY

Datasets ready for analysis:
  ✓ cgm          |  65,535 records | 2025-04-16 11:17:05 to 2027-07-03 23:56:44
  ✓ finger       |   1,691 records | 2025-04-16 11:05:34 to 2027-07-03 23:31:44
  ✓ meal         |     702 records | 2025-04-16 18:00:00 to 2027-07-03 19:30:00
  ✓ basal        |     357 records | 2025-04-16 00:00:00 to 2027-06-22 14:00:00
  ✓ exercise     |      61 records | 2025-04-17 18:30:00 to 2027-07-03 09:45:00

Datasets excluded (parsing issues):
  ✗ bolus       | All timestamps missing (NaT) - skip for now
  ✗ sleep       | All timest

---
# PART 2: Vertical Stack - Align Events with CGM Timeline
---

## Overview
Merge meals, exercise, and other events with CGM using **merge_asof**:
- For each glucose reading, find the most recent meal, exercise, or other event
- This creates a rich feature space without duplicating CGM rows
- Must execute BEFORE hypoglycemia labeling so labels are created on full feature set

(Ideal for aligning two time series that don't share exact timestamps)

In [6]:
### 2.1 Merge Meals with CGM Using merge_asof

meal = datasets_to_use['meal']

meal_renamed = meal[['patient_id', 'ts', 'carbs']].copy()
meal_renamed.columns = ['patient_id', 'meal_ts', 'meal_carbs']

cgm_with_meals = pd.merge_asof(
    cgm_clean.sort_values('ts'),
    meal_renamed.sort_values('meal_ts'),
    left_on='ts',
    right_on='meal_ts',
    by='patient_id',
    direction='backward',
    tolerance=pd.Timedelta(hours=4)
)

cgm_with_meals['minutes_since_meal'] = (cgm_with_meals['ts'] - cgm_with_meals['meal_ts']).dt.total_seconds() / 60
cgm_with_meals['meal_carbs'] = cgm_with_meals['meal_carbs'].fillna(0)
cgm_with_meals['minutes_since_meal'] = cgm_with_meals['minutes_since_meal'].fillna(999)

print("✓ Meals merged with CGM")
print(f"  Readings with meal data: {(cgm_with_meals['minutes_since_meal'] < 999).sum()}")
print(f"  Avg carbs when meal present: {cgm_with_meals[cgm_with_meals['meal_carbs'] > 0]['meal_carbs'].mean():.1f}g\n")

### 2.2 Merge Exercise with CGM

exercise = datasets_to_use['exercise']

exercise_renamed = exercise[['patient_id', 'ts', 'intensity', 'duration']].copy()
exercise_renamed.columns = ['patient_id', 'exercise_ts', 'exercise_intensity', 'exercise_duration']

cgm_with_events = pd.merge_asof(
    cgm_with_meals.sort_values('ts'),
    exercise_renamed.sort_values('exercise_ts'),
    left_on='ts',
    right_on='exercise_ts',
    by='patient_id',
    direction='backward',
    tolerance=pd.Timedelta(hours=3)
)

cgm_with_events['minutes_since_exercise'] = (cgm_with_events['ts'] - cgm_with_events['exercise_ts']).dt.total_seconds() / 60
cgm_with_events['exercise_intensity'] = cgm_with_events['exercise_intensity'].fillna(0)
cgm_with_events['exercise_duration'] = cgm_with_events['exercise_duration'].fillna(0)
cgm_with_events['minutes_since_exercise'] = cgm_with_events['minutes_since_exercise'].fillna(999)

print("✓ Exercise merged with CGM")
print(f"  Readings with exercise data: {(cgm_with_events['minutes_since_exercise'] < 999).sum()}")
print(f"  Avg intensity: {cgm_with_events[cgm_with_events['exercise_intensity'] > 0]['exercise_intensity'].mean():.1f}\n")

### 2.3 Summary

cgm_stacked = cgm_with_events.copy()

print("="*70)
print("PART 2 COMPLETE: Full Dataset with Events Aligned")
print("="*70)
print(f"\nShape: {cgm_stacked.shape}")
print(f"Columns: {list(cgm_stacked.columns)}\n")
print("Sample rows:")
print(cgm_stacked[['patient_id', 'ts', 'value', 'meal_carbs', 'minutes_since_meal', 'exercise_intensity']].head(10))

print(f"\n✅ Part 2 complete: Ready for hypoglycemia labeling on full feature set")

✓ Meals merged with CGM
  Readings with meal data: 23941
  Avg carbs when meal present: 52.0g

✓ Exercise merged with CGM
  Readings with exercise data: 1573
  Avg intensity: 5.8

PART 2 COMPLETE: Full Dataset with Events Aligned

Shape: (65535, 10)
Columns: ['patient_id', 'ts', 'value', 'meal_ts', 'meal_carbs', 'minutes_since_meal', 'exercise_ts', 'exercise_intensity', 'exercise_duration', 'minutes_since_exercise']

Sample rows:
        patient_id                  ts  value  meal_carbs  minutes_since_meal  \
0  552-ws-training 2025-04-16 11:17:05   95.0         0.0               999.0   
1  552-ws-training 2025-04-16 11:22:05   86.0         0.0               999.0   
2  552-ws-training 2025-04-16 11:27:05   81.0         0.0               999.0   
3  552-ws-training 2025-04-16 11:32:05   81.0         0.0               999.0   
4  552-ws-training 2025-04-16 11:37:05   82.0         0.0               999.0   
5  552-ws-training 2025-04-16 11:42:05   82.0         0.0               999.0   

---
# PART 3: Hypoglycemia Labeling & 30-Minute Lookahead
---

## Overview
- Label each glucose reading: hypoglycemic (< 70 mg/dL) or euglycemic (>= 70 mg/dL)
- Create 30-minute lookahead labels: "Will this patient be hypoglycemic in the next 30 minutes?"
- This creates a binary classification task: predict hypoglycemia onset
- Created on stacked dataset with meal/exercise context

In [7]:
def create_hypoglycemia_labels(cgm_df, lookahead_minutes=30):
    """Create binary labels for hypoglycemia prediction task."""
    
    result = []
    
    for patient_id in cgm_df['patient_id'].unique():
        patient_data = cgm_df[cgm_df['patient_id'] == patient_id].sort_values('ts').copy()
        
        patient_data['is_hypoglycemic'] = (patient_data['value'] < 70).astype(int)
        patient_data['will_be_hypoglycemic_30min'] = 0
        
        lookahead_delta = pd.Timedelta(minutes=lookahead_minutes)
        
        for idx, row in patient_data.iterrows():
            current_time = row['ts']
            window_end = current_time + lookahead_delta
            
            future_readings = patient_data[
                (patient_data['ts'] > current_time) & 
                (patient_data['ts'] <= window_end)
            ]
            
            if len(future_readings) > 0 and (future_readings['value'] < 70).any():
                patient_data.loc[idx, 'will_be_hypoglycemic_30min'] = 1
        
        result.append(patient_data)
    
    return pd.concat(result, ignore_index=True)

cgm_labeled = create_hypoglycemia_labels(cgm_stacked)

print("="*70)
print("HYPOGLYCEMIA LABELING COMPLETE")
print("="*70)

print("\n📊 CLASS DISTRIBUTION")
print("-"*70)
print(f"\nCurrent hypoglycemia (< 70 mg/dL):")
print(cgm_labeled['is_hypoglycemic'].value_counts())
print(f"  Prevalence: {cgm_labeled['is_hypoglycemic'].mean() * 100:.2f}%")

print(f"\n30-minute lookahead (will be < 70 in next 30 min):")
print(cgm_labeled['will_be_hypoglycemic_30min'].value_counts())
print(f"  Prevalence: {cgm_labeled['will_be_hypoglycemic_30min'].mean() * 100:.2f}%")

print(f"\n📋 PER-PATIENT HYPOGLYCEMIA PREVALENCE")
print("-"*70)
per_patient_stats = cgm_labeled.groupby('patient_id').agg({
    'is_hypoglycemic': ['count', 'sum', 'mean'],
    'will_be_hypoglycemic_30min': ['sum', 'mean']
}).round(4)

per_patient_stats.columns = ['total_readings', 'hypo_count', 'hypo_rate', 'upcoming_hypo_count', 'upcoming_hypo_rate']
print(per_patient_stats)

cgm_final = cgm_labeled.copy()
print(f"\n✅ Part 3 complete: Labels created on full feature-enriched dataset")
print(f"✅ Ready for Part 4: Temporal feature engineering")

HYPOGLYCEMIA LABELING COMPLETE

📊 CLASS DISTRIBUTION
----------------------------------------------------------------------

Current hypoglycemia (< 70 mg/dL):
is_hypoglycemic
0    63311
1     2224
Name: count, dtype: int64
  Prevalence: 3.39%

30-minute lookahead (will be < 70 in next 30 min):
will_be_hypoglycemic_30min
0    62208
1     3327
Name: count, dtype: int64
  Prevalence: 5.08%

📋 PER-PATIENT HYPOGLYCEMIA PREVALENCE
----------------------------------------------------------------------
                 total_readings  hypo_count  hypo_rate  upcoming_hypo_count  \
patient_id                                                                    
540-ws-training           11947         759     0.0635                 1168   
544-ws-training           10623         146     0.0137                  236   
552-ws-training            9080         311     0.0343                  489   
567-ws-training           10858         695     0.0640                  840   
584-ws-training          

---
# PART 4: Feature Engineering & Temporal Aggregation
---

## Overview
Create rolling window features to capture glucose dynamics:
- **Trend:** Glucose slope over last 15, 30 minutes
- **Volatility:** Standard deviation of glucose
- **Acceleration:** Rate of change of rate of change
- **Context:** Fasting vs. fed state indicator

In [8]:
def create_temporal_features(df, window_sizes=[3, 6, 12]):
    """Create temporal features for each glucose reading using rolling windows."""

    df_features = df.copy()

    # Pre-create all feature columns
    for window in window_sizes:
        minutes = window * 5
        df_features[f'glucose_slope_{minutes}min'] = np.nan
        df_features[f'glucose_volatility_{minutes}min'] = np.nan
        df_features[f'glucose_mean_{minutes}min'] = np.nan

    df_features['glucose_acceleration'] = np.nan
    df_features['is_fasting'] = 0

    # Process each patient
    for patient_id in df_features['patient_id'].unique():
        patient_mask = df_features['patient_id'] == patient_id
        patient_indices = df_features[patient_mask].index

        glucose = df_features.loc[patient_mask, 'value'].values

        # Calculate temporal features for each window size
        for window in window_sizes:
            minutes = window * 5

            # Glucose slope (rate of change)
            trend_col = f'glucose_slope_{minutes}min'
            slopes = np.full(len(glucose), np.nan)
            for i in range(window, len(glucose)):
                if not np.isnan(glucose[i-window:i]).any():
                    slopes[i] = (glucose[i] - glucose[i-window]) / window
            df_features.loc[patient_indices, trend_col] = slopes

            # Glucose volatility (rolling std dev)
            vol_col = f'glucose_volatility_{minutes}min'
            vol_values = df_features.loc[patient_mask, 'value'].rolling(window=window, min_periods=1).std().values
            df_features.loc[patient_indices, vol_col] = vol_values

            # Glucose mean
            mean_col = f'glucose_mean_{minutes}min'
            mean_values = df_features.loc[patient_mask, 'value'].rolling(window=window, min_periods=1).mean().values
            df_features.loc[patient_indices, mean_col] = mean_values

        # Glucose acceleration (second derivative)
        slope_vals = df_features.loc[patient_mask, f'glucose_slope_15min'].values
        accel = np.gradient(slope_vals, edge_order=2)
        df_features.loc[patient_indices, 'glucose_acceleration'] = accel

        # Fasting state indicator
        is_fasting = ((df_features.loc[patient_mask, 'value'] < 100) & 
                      (df_features.loc[patient_mask, 'minutes_since_meal'] > 120)).astype(int).values
        df_features.loc[patient_indices, 'is_fasting'] = is_fasting

    return df_features

print("Creating temporal features...")
cgm_features = create_temporal_features(cgm_final)

print("="*70)
print("PART 4: TEMPORAL FEATURES CREATED")
print("="*70)

# Debug: Check what columns exist
all_cols = cgm_features.columns.tolist()
print(f"\nTotal columns in cgm_features: {len(all_cols)}")
print(f"Columns: {all_cols}\n")

feature_cols = [col for col in cgm_features.columns if any(x in col for x in ['slope', 'volatility', 'mean', 'acceleration', 'fasting'])]
print(f"New features created ({len(feature_cols)}):")
for col in sorted(feature_cols):
    print(f"  - {col}")

if len(feature_cols) > 0:
    print(f"\nFeature statistics:")
    stat_cols = [col for col in feature_cols if 'slope' in col or 'volatility' in col]
    if stat_cols:
        print(cgm_features[stat_cols].describe())
else:
    print("\nWARNING: No temporal features found!")

print(f"\n✅ Part 4 complete: {cgm_features.shape[1]} total columns")

Creating temporal features...
PART 4: TEMPORAL FEATURES CREATED

Total columns in cgm_features: 23
Columns: ['patient_id', 'ts', 'value', 'meal_ts', 'meal_carbs', 'minutes_since_meal', 'exercise_ts', 'exercise_intensity', 'exercise_duration', 'minutes_since_exercise', 'is_hypoglycemic', 'will_be_hypoglycemic_30min', 'glucose_slope_15min', 'glucose_volatility_15min', 'glucose_mean_15min', 'glucose_slope_30min', 'glucose_volatility_30min', 'glucose_mean_30min', 'glucose_slope_60min', 'glucose_volatility_60min', 'glucose_mean_60min', 'glucose_acceleration', 'is_fasting']

New features created (11):
  - glucose_acceleration
  - glucose_mean_15min
  - glucose_mean_30min
  - glucose_mean_60min
  - glucose_slope_15min
  - glucose_slope_30min
  - glucose_slope_60min
  - glucose_volatility_15min
  - glucose_volatility_30min
  - glucose_volatility_60min
  - is_fasting

Feature statistics:
       glucose_slope_15min  glucose_volatility_15min  glucose_slope_30min  \
count         65517.000000     

---
# PART 5: Shapelet Discovery & Feature Extraction
---

## Overview
A **shapelet** is a discriminative subsequence that separates two classes.
1. Extract all glucose subsequences (windows of various lengths)
2. Compute correlation with future hypoglycemia
3. Rank by discriminative power
4. Keep top-k shapelets for the classifier

In [9]:
def extract_shapelets(df, target_col='will_be_hypoglycemic_30min', shapelet_lengths=[5, 10, 15], top_k=3):
    """Discover discriminative shapelets for hypoglycemia prediction."""
    
    shapelets_list = []
    
    for length in shapelet_lengths:
        for patient_id in df['patient_id'].unique():
            patient_df = df[df['patient_id'] == patient_id].reset_index(drop=True)
            glucose = patient_df['value'].values
            target = patient_df[target_col].values
            
            for start in range(len(glucose) - length):
                shapelet_pattern = glucose[start:start+length]
                
                if np.isnan(shapelet_pattern).any():
                    continue
                
                shapelet_normalized = (shapelet_pattern - shapelet_pattern.mean()) / (shapelet_pattern.std() + 1e-8)
                
                correlation_values = []
                for i in range(start, min(start+length+1, len(target))):
                    if not np.isnan(shapelet_normalized).any():
                        shapelet_mean = shapelet_normalized.mean()
                        label_at_i = target[i]
                        correlation_values.append(shapelet_mean * label_at_i)
                
                if correlation_values:
                    avg_correlation = np.mean(correlation_values)
                    shapelets_list.append({
                        'length': length,
                        'patient_id': patient_id,
                        'start_idx': start,
                        'shapelet': shapelet_pattern,
                        'shapelet_normalized': shapelet_normalized,
                        'correlation_score': abs(avg_correlation)
                    })
    
    df_shapelets = pd.DataFrame(shapelets_list)
    df_shapelets = df_shapelets.sort_values('correlation_score', ascending=False)
    
    top_shapelets = []
    for length in shapelet_lengths:
        length_df = df_shapelets[df_shapelets['length'] == length].head(top_k)
        top_shapelets.extend(length_df.to_dict('records'))
    
    return {
        'shapelets': top_shapelets,
        'all_scores': df_shapelets
    }

print("Discovering shapelets... (this may take a minute)")
shapelet_results = extract_shapelets(cgm_features, top_k=3)

print("\n" + "="*70)
print("PART 5: TOP SHAPELETS DISCOVERED")
print("="*70)

print(f"\nTop-K Shapelets (k=3 per length):\n")
for i, shapelet_data in enumerate(shapelet_results['shapelets'], 1):
    length = shapelet_data['length']
    minutes = length * 5
    score = shapelet_data['correlation_score']
    pattern = shapelet_data['shapelet']
    patient = shapelet_data['patient_id']
    
    print(f"{i}. Length={minutes}min | Score={score:.4f} | Patient={patient}")
    print(f"   Pattern (glucose, mg/dL): {' → '.join([f'{v:.0f}' for v in pattern])}")
    print()

print("\n📊 Shapelet Summary:")
print(f"  Total shapelet candidates: {len(shapelet_results['all_scores'])}")
print(f"  Shapelets selected: {len(shapelet_results['shapelets'])}")
print(f"  Lengths analyzed: {shapelet_results['all_scores']['length'].unique()}")

print(f"\n✅ Part 5 complete: Shapelets ready for feature engineering")

Discovering shapelets... (this may take a minute)

PART 5: TOP SHAPELETS DISCOVERED

Top-K Shapelets (k=3 per length):

1. Length=25min | Score=0.0000 | Patient=544-ws-training
   Pattern (glucose, mg/dL): 65 → 65 → 64 → 64 → 64

2. Length=25min | Score=0.0000 | Patient=567-ws-training
   Pattern (glucose, mg/dL): 65 → 65 → 64 → 65 → 64

3. Length=25min | Score=0.0000 | Patient=540-ws-training
   Pattern (glucose, mg/dL): 68 → 68 → 69 → 69 → 68

4. Length=50min | Score=0.0000 | Patient=540-ws-training
   Pattern (glucose, mg/dL): 68 → 67 → 67 → 69 → 69 → 68 → 68 → 68 → 68 → 69

5. Length=50min | Score=0.0000 | Patient=540-ws-training
   Pattern (glucose, mg/dL): 69 → 68 → 67 → 67 → 69 → 69 → 68 → 68 → 68 → 68

6. Length=50min | Score=0.0000 | Patient=567-ws-training
   Pattern (glucose, mg/dL): 58 → 58 → 57 → 57 → 57 → 57 → 57 → 57 → 57 → 57

7. Length=75min | Score=0.0000 | Patient=540-ws-training
   Pattern (glucose, mg/dL): 67 → 69 → 69 → 68 → 68 → 68 → 68 → 69 → 70 → 70 → 70 → 70 →

---
# PART 6: Binary Classifiers & Comparative Evaluation
---

## Overview
1. **Baseline (Logistic Regression):** Engineered temporal features + shapelet similarity scores
2. **Neural Network:** Raw glucose sequences as baseline
3. **Comparison:** AUROC, confusion matrices, per-patient performance

In [10]:
def compute_shapelet_features(df, shapelets_list, target_col='will_be_hypoglycemic_30min'):
    """Compute similarity to each discovered shapelet."""
    
    # Start with basic info AND temporal features
    temporal_cols = [col for col in df.columns if any(x in col for x in ['slope', 'volatility', 'mean', 'acceleration', 'fasting'])]
    
    shapelet_features = df[['patient_id', 'ts', target_col] + temporal_cols].copy()
    
    for idx, shapelet_data in enumerate(shapelets_list):
        shapelet_pattern = shapelet_data['shapelet_normalized']
        length = len(shapelet_pattern)
        
        feature_col = f'shapelet_sim_{idx}'
        shapelet_features[feature_col] = np.nan
        
        for patient_id in df['patient_id'].unique():
            mask = df['patient_id'] == patient_id
            glucose = df.loc[mask, 'value'].values
            
            glucose_norm = (glucose - glucose.mean()) / (glucose.std() + 1e-8)
            
            similarities = []
            for i in range(len(glucose_norm) - length + 1):
                window = glucose_norm[i:i+length]
                if not np.isnan(window).any():
                    corr = np.corrcoef(shapelet_pattern, window)[0, 1]
                    similarities.append(corr if not np.isnan(corr) else 0)
                else:
                    similarities.append(0)
            
            if similarities:
                similarities_array = np.array(similarities + [0] * length)[:len(glucose_norm)]
                shapelet_features.loc[mask, feature_col] = similarities_array
    
    shapelet_features = shapelet_features.fillna(0)
    return shapelet_features

print("Computing shapelet-based features...")
shapelet_feats = compute_shapelet_features(cgm_features, shapelet_results['shapelets'])

print(f"✓ Shapelet features: {shapelet_feats.shape[1] - 3} similarity scores per reading")
print(f"✓ Temporal features included: {len([col for col in shapelet_feats.columns if any(x in col for x in ['slope', 'volatility', 'mean', 'acceleration', 'fasting'])])} columns\n")

# Prepare data
feature_cols_lr = ([col for col in shapelet_feats.columns if any(x in col for x in ['slope', 'volatility', 'mean', 'acceleration', 'fasting'])] +
                    [col for col in shapelet_feats.columns if 'shapelet_sim' in col])

print(f"✓ Total features for Logistic Regression: {len(feature_cols_lr)}")

X_logistic = shapelet_feats[feature_cols_lr].fillna(0).values
y = cgm_features['will_be_hypoglycemic_30min'].values

X_lstm_sequences = []
y_lstm = []

for patient_id in cgm_features['patient_id'].unique():
    mask = cgm_features['patient_id'] == patient_id
    glucose_seq = cgm_features.loc[mask, 'value'].values
    target_seq = cgm_features.loc[mask, 'will_be_hypoglycemic_30min'].values
    
    seq_len = 12
    for i in range(seq_len, len(glucose_seq)):
        X_lstm_sequences.append(glucose_seq[i-seq_len:i])
        y_lstm.append(target_seq[i])

X_lstm_sequences = np.array(X_lstm_sequences)
y_lstm = np.array(y_lstm)

print(f"✓ Logistic Regression: {X_logistic.shape[0]} samples × {X_logistic.shape[1]} features")
print(f"✓ LSTM: {X_lstm_sequences.shape[0]} sequences × {X_lstm_sequences.shape[1]} timesteps")
print(f"✓ Target: {y.sum()} positive examples (prevalence: {y.mean()*100:.2f}%)\n")

Computing shapelet-based features...
✓ Shapelet features: 20 similarity scores per reading
✓ Temporal features included: 11 columns

✓ Total features for Logistic Regression: 20
✓ Logistic Regression: 65535 samples × 20 features
✓ LSTM: 65463 sequences × 12 timesteps
✓ Target: 3327 positive examples (prevalence: 5.08%)



In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

# Create index tracking for train_test_split
all_indices = np.arange(len(X_logistic))

# Train Logistic Regression
X_train_lr, X_test_lr, y_train, y_test, train_idx, test_idx = train_test_split(
    X_logistic, y, all_indices, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_lr_scaled = scaler.fit_transform(X_train_lr)
X_test_lr_scaled = scaler.transform(X_test_lr)

lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_lr_scaled, y_train)

y_pred_lr_train = lr_model.predict_proba(X_train_lr_scaled)[:, 1]
y_pred_lr_test = lr_model.predict_proba(X_test_lr_scaled)[:, 1]

lr_train_auroc = roc_auc_score(y_train, y_pred_lr_train)
lr_test_auroc = roc_auc_score(y_test, y_pred_lr_test)

print("="*70)
print("LOGISTIC REGRESSION (SHAPELET-BASED) RESULTS")
print("="*70)
print(f"Train AUROC: {lr_train_auroc:.4f}")
print(f"Test AUROC:  {lr_test_auroc:.4f}")

y_pred_lr_binary = (y_pred_lr_test > 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr_binary).ravel()
print(f"\nConfusion Matrix (threshold=0.5):")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")
print(f"\nSensitivity (Recall): {tp/(tp+fn):.4f}")
print(f"Specificity:         {tn/(tn+fp):.4f}")

print(f"\n✅ Logistic Regression model trained and evaluated")

LOGISTIC REGRESSION (SHAPELET-BASED) RESULTS
Train AUROC: 0.9902
Test AUROC:  0.9896

Confusion Matrix (threshold=0.5):
  True Negatives:  11698
  False Positives: 744
  False Negatives: 15
  True Positives:  650

Sensitivity (Recall): 0.9774
Specificity:         0.9402

✅ Logistic Regression model trained and evaluated


In [12]:
# Train Neural Network (MLP on sequences)
all_lstm_indices = np.arange(len(X_lstm_sequences))

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm, _, _ = train_test_split(
    X_lstm_sequences, y_lstm, all_lstm_indices, test_size=0.2, random_state=42, stratify=y_lstm
)

scaler_lstm = StandardScaler()
X_train_lstm_scaled = scaler_lstm.fit_transform(X_train_lstm)
X_test_lstm_scaled = scaler_lstm.transform(X_test_lstm)

lstm_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
lstm_model.fit(X_train_lstm_scaled, y_train_lstm)

y_pred_lstm_train = lstm_model.predict_proba(X_train_lstm_scaled)[:, 1]
y_pred_lstm_test = lstm_model.predict_proba(X_test_lstm_scaled)[:, 1]

lstm_train_auroc = roc_auc_score(y_train_lstm, y_pred_lstm_train)
lstm_test_auroc = roc_auc_score(y_test_lstm, y_pred_lstm_test)

print("="*70)
print("NEURAL NETWORK (RAW GLUCOSE SEQUENCES) RESULTS")
print("="*70)
print(f"Train AUROC: {lstm_train_auroc:.4f}")
print(f"Test AUROC:  {lstm_test_auroc:.4f}")

y_pred_lstm_binary = (y_pred_lstm_test > 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_lstm, y_pred_lstm_binary).ravel()
print(f"\nConfusion Matrix (threshold=0.5):")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")
print(f"\nSensitivity (Recall): {tp/(tp+fn):.4f}")
print(f"Specificity:         {tn/(tn+fp):.4f}")

NEURAL NETWORK (RAW GLUCOSE SEQUENCES) RESULTS
Train AUROC: 0.9735
Test AUROC:  0.9740

Confusion Matrix (threshold=0.5):
  True Negatives:  12297
  False Positives: 134
  False Negatives: 228
  True Positives:  434

Sensitivity (Recall): 0.6556
Specificity:         0.9892


In [13]:
# Model Comparison
print("\n" + "="*70)
print("MODEL COMPARISON: SHAPELETS vs RAW SEQUENCES")
print("="*70)

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression (Shapelets)', 'Neural Network (Raw Glucose)'],
    'Train AUROC': [lr_train_auroc, lstm_train_auroc],
    'Test AUROC': [lr_test_auroc, lstm_test_auroc],
    'Features': [len(feature_cols_lr), 12]
})

print("\n", comparison_df.to_string(index=False))

if lr_test_auroc > lstm_test_auroc:
    print(f"\n🏆 WINNER: Logistic Regression (+{(lr_test_auroc - lstm_test_auroc):.4f} AUROC)")
    print("   → Shapelet-based features are more effective for this task")
else:
    print(f"\n🏆 WINNER: Neural Network (+{(lstm_test_auroc - lr_test_auroc):.4f} AUROC)")
    print("   → Raw glucose sequences capture discriminative patterns better")

# Per-patient analysis using original indices
print("\n" + "="*70)
print("PER-PATIENT AUROC ANALYSIS")
print("="*70)

per_patient_auroc = []

for patient_id in cgm_features['patient_id'].unique():
    # Get patient mask for test indices
    patient_mask = cgm_features.iloc[test_idx]['patient_id'] == patient_id
    
    if patient_mask.sum() > 10:
        y_patient = y_test[patient_mask.values]
        y_pred_patient = y_pred_lr_test[patient_mask.values]
        
        if len(np.unique(y_patient)) == 2:
            patient_auroc = roc_auc_score(y_patient, y_pred_patient)
            per_patient_auroc.append({
                'patient_id': patient_id,
                'auroc': patient_auroc,
                'n_samples': patient_mask.sum(),
                'hypo_rate': y_patient.mean()
            })

if per_patient_auroc:
    df_per_patient = pd.DataFrame(per_patient_auroc).sort_values('auroc', ascending=False)
    print("\n", df_per_patient.to_string(index=False))
    print(f"\nMean per-patient AUROC: {df_per_patient['auroc'].mean():.4f}")
    print(f"Std:  {df_per_patient['auroc'].std():.4f}")

    print("\n📊 Insight: Patients may have different prediction difficulty")
    print("   Consider per-patient model tuning for challenging patients")
else:
    print("\nNote: Insufficient per-patient samples for separate AUROC calculation")

print(f"\n✅ PART 6 COMPLETE: All models evaluated!")
print(f"\n" + "="*70)
print(f"PROJECT SUMMARY")
print(f"="*70)
print(f"""
✓ Part 1: Loaded & cleaned 65,535 CGM readings from 6 patients
✓ Part 2: Vertically stacked meals, exercise events with CGM
✓ Part 3: Created hypoglycemia labels ({(cgm_features['will_be_hypoglycemic_30min'].mean()*100):.2f}% prevalence)
✓ Part 4: Engineered 11 temporal features (slopes, volatility, etc.)
✓ Part 5: Discovered {len(shapelet_results['shapelets'])} discriminative shapelets
✓ Part 6: Built & compared binary classifiers

FINAL RESULTS:
  - Shapelet-based classifier AUROC: {lr_test_auroc:.4f}
  - Raw glucose classifier AUROC:    {lstm_test_auroc:.4f}
  - Best model: {'Shapelets' if lr_test_auroc > lstm_test_auroc else 'Raw sequences'}
  
NEXT STEPS:
  1. Optimize hyperparameters (cross-validation)
  2. Test on held-out patients (external validation)
  3. Implement alarm lead-time analysis
  4. Deploy for prospective evaluation
""")


MODEL COMPARISON: SHAPELETS vs RAW SEQUENCES

                           Model  Train AUROC  Test AUROC  Features
Logistic Regression (Shapelets)     0.990191    0.989583        20
   Neural Network (Raw Glucose)     0.973464    0.973975        12

🏆 WINNER: Logistic Regression (+0.0156 AUROC)
   → Shapelet-based features are more effective for this task

PER-PATIENT AUROC ANALYSIS

      patient_id    auroc  n_samples  hypo_rate
544-ws-training 0.996539       2094   0.022445
567-ws-training 0.995393       2222   0.075158
552-ws-training 0.993886       1839   0.058728
596-ws-training 0.989155       2144   0.039646
540-ws-training 0.983606       2345   0.096375
584-ws-training 0.958029       2463   0.012992

Mean per-patient AUROC: 0.9861
Std:  0.0146

📊 Insight: Patients may have different prediction difficulty
   Consider per-patient model tuning for challenging patients

✅ PART 6 COMPLETE: All models evaluated!

PROJECT SUMMARY

✓ Part 1: Loaded & cleaned 65,535 CGM readings from 6 